# Notebook 4: Đánh Giá Định Lượng Toàn Diện 20 Ca Bệnh (Benchmark Evaluation)
**Đồ án Xử lý ảnh số (2026–2027) — Khoa CNTT — HUIT**  
**Nhóm sinh viên:** Hà Mạnh Trường (2045240326) & Huỳnh Tâm Trí (2045240315)  
**Phụ trách chính:** Huỳnh Tâm Trí

---
### Tiêu chí định lượng y khoa:
1. **Dice Similarity Coefficient (DSC / F1):** Độ trùng lặp diện tích u.
2. **Jaccard Index (IoU):** Tỷ lệ giao trên hợp.
3. **Sensitivity (Độ nhạy):** Khả năng không bỏ sót pixel u não.
4. **Specificity (Độ đặc hiệu):** Khả năng không nhận nhầm mô lành thành u.
5. **Độ trễ trung bình (Latency ms/ảnh).**


In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

sys.path.append(os.path.abspath(".."))
from src.preprocessing.skull_strip import strip_skull
from src.segmentation.watershed import segment_watershed
from src.segmentation.region_growing import segment_region_growing
from src.evaluation.benchmark import run_benchmark


### 1. Thực Thi Benchmark Trên Toàn Bộ 20 Mẫu Ảnh


In [ ]:
img_dir = "../data/sample/images"
mask_dir = "../data/sample/masks"

# Hàm wrapper cho Watershed
def predict_watershed(img):
    stripped, _ = strip_skull(img)
    return segment_watershed(stripped)

# Hàm wrapper cho Region Growing
def predict_region_growing(img):
    stripped, _ = strip_skull(img)
    return segment_region_growing(stripped)

print("Đang chạy benchmark Watershed...")
df_ws = run_benchmark(img_dir, mask_dir, predict_watershed, method_name="Watershed")

print("Đang chạy benchmark Region Growing...")
df_rg = run_benchmark(img_dir, mask_dir, predict_region_growing, method_name="Region Growing")


### 2. Bảng Tổng Hợp So Sánh Thống Kê (Mean ± Std)


In [ ]:
summary = pd.DataFrame({
    "Chỉ Số Đánh Giá": ["Dice Score (DSC)", "IoU (Jaccard)", "Sensitivity (Độ nhạy)", "Specificity (Đặc hiệu)", "Thời gian xử lý (ms)"],
    "Watershed (Baseline)": [
        f"{df_ws['dice'].mean():.4f} ± {df_ws['dice'].std():.4f}",
        f"{df_ws['iou'].mean():.4f} ± {df_ws['iou'].std():.4f}",
        f"{df_ws['sensitivity'].mean():.4f} ± {df_ws['sensitivity'].std():.4f}",
        f"{df_ws['specificity'].mean():.4f} ± {df_ws['specificity'].std():.4f}",
        f"{df_ws['latency_ms'].mean():.2f} ± {df_ws['latency_ms'].std():.2f} ms"
    ],
    "Region Growing (Đối chứng)": [
        f"{df_rg['dice'].mean():.4f} ± {df_rg['dice'].std():.4f}",
        f"{df_rg['iou'].mean():.4f} ± {df_rg['iou'].std():.4f}",
        f"{df_rg['sensitivity'].mean():.4f} ± {df_rg['sensitivity'].std():.4f}",
        f"{df_rg['specificity'].mean():.4f} ± {df_rg['specificity'].std():.4f}",
        f"{df_rg['latency_ms'].mean():.2f} ± {df_rg['latency_ms'].std():.2f} ms"
    ]
})

summary


### 3. Biểu Đồ Cột So Sánh Điểm Dice Trên Từng Ca Bệnh


In [ ]:
x = np.arange(len(df_ws))
width = 0.38

fig, ax = plt.subplots(figsize=(16, 6))
rects1 = ax.bar(x - width/2, df_ws['dice'], width, label='Watershed (Baseline)', color='#3B82F6', alpha=0.85)
rects2 = ax.bar(x + width/2, df_rg['dice'], width, label='Region Growing (Đối chứng)', color='#10B981', alpha=0.85)

ax.set_ylabel('Điểm Dice Similarity Coefficient', fontsize=12)
ax.set_title('So Sánh Điểm Dice Score Giữa 2 Phương Pháp Trên 20 Ca Bệnh TCGA-LGG', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([f"Ca {i+1:02d}" for i in range(len(df_ws))], rotation=45)
ax.legend(fontsize=12)
ax.set_ylim(0, 1.05)
ax.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()
